In [1]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()
print("VADER Sentiment Analyzer initialized successfully!")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\praful\AppData\Roaming\nltk_data...


VADER Sentiment Analyzer initialized successfully!


In [2]:
reddit_df = pd.read_csv(r"C:\Users\praful\Desktop\madhulika\sentiment stock analysis\data\gme_reddit_clean.csv")
print(f"Successfully loaded {reddit_df.shape[0]} rows and {reddit_df.shape[1]} columns!")
print("\nFirst 3 rows preview:")
reddit_df.head(3)

Successfully loaded 106930 rows and 5 columns!

First 3 rows preview:


,id,title,score,date,num_comments
0,ll0n4p,"Need explanations on Level 2 data for GME, why...",1,2021-02-16,2
1,ll0d5c,SAS = GME,1,2021-02-16,2
2,lkzzpj,Is anybody still holding onto GME?,1,2021-02-16,2


In [3]:
# Testing text mimicking 2021 Reddit hype
sample_text = "GME TO THE MOON!!! 🚀 Buy the dip and hold, do not sell!!"

#sentiment score dictionary
sample_score = sia.polarity_scores(sample_text)

print(f"Sample Post: {sample_text}")
print(f"VADER Output: {sample_score}")

Sample Post: GME TO THE MOON!!! 🚀 Buy the dip and hold, do not sell!!
VADER Output: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [4]:
# Extracting only the compound sentiment score for every title
reddit_df['sentiment_score'] = reddit_df['title'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

def categorize_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

reddit_df['sentiment_type'] = reddit_df['sentiment_score'].apply(categorize_sentiment)

print("Scoring complete! Let's see the breakdown:")
print(reddit_df['sentiment_type'].value_counts())
print("\nPreview of the newly scored data:")
reddit_df[['title', 'sentiment_score', 'sentiment_type']].head(5)

Scoring complete! Let's see the breakdown:
sentiment_type
Neutral     53587
Positive    31319
Negative    22024
Name: count, dtype: int64

Preview of the newly scored data:


,title,sentiment_score,sentiment_type
0,"Need explanations on Level 2 data for GME, why...",0.0,Neutral
1,SAS = GME,0.0,Neutral
2,Is anybody still holding onto GME?,0.0,Neutral
3,Should I sell GameStop stock?,0.0,Neutral
4,Show is over for GME and AMC? What happened to...,0.0,Neutral


In [5]:
# Group by date and calculating the mean sentiment and total posts for each day
daily_sentiment = reddit_df.groupby('date').agg(
    mean_sentiment=('sentiment_score', 'mean'),
    total_posts=('id', 'count')
).reset_index()

print("Daily sentiment aggregation complete!")
print(f"Aggregated data contains {daily_sentiment.shape[0]} unique days.")
daily_sentiment.head(5)

Daily sentiment aggregation complete!
Aggregated data contains 497 unique days.


,date,mean_sentiment,total_posts
0,2012-06-01,0.0000,1
1,2013-05-09,0.0000,1
2,2014-11-21,0.0000,1
3,2014-12-10,0.0000,1
4,2015-05-29,0.4404,1


In [6]:
reddit_df.to_csv(r"C:\Users\praful\Desktop\madhulika\sentiment stock analysis\data\gme_reddit_scored.csv", index=False)

daily_sentiment.to_csv(r"C:\Users\praful\Desktop\madhulika\sentiment stock analysis\data\gme_reddit_daily_sentiment.csv", index=False)

print("Both CSV files exported successfully to your data directory!")

Both CSV files exported successfully to your data directory!
